# A3: FAO – Natural Earth country-name alignment

In [ ]:
import yaml
from pathlib import Path
import sys
sys.path.append(str(Path('../src')))

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr


with open(Path('../config.yaml')) as f:
    cfg = yaml.safe_load(f)

In [ ]:
for k, v in cfg['paths'].items():
    cfg["paths"][k] = Path(v)#.expanduser().resolve()

cfg["data_root"] = Path(cfg["data_root"])

# Clean administrative names

In [ ]:
import country_converter as coco
import pandas as pd
cc = coco.CountryConverter()

## Clean Natural Earth data: keep necessary columns and rename

In [ ]:
gdf_world_ne = gpd.read_file(cfg["data_root"] / "nature earth/ne_10m_admin_0_countries_chn.shp", encoding='utf-8')
hk_mc = gdf_world_ne[gdf_world_ne['ISO_A3_EH'].isin(['HKG', 'MAC'])]
CHINA_ISO3 = {'HKG', 'MAC'} 

def assign_iso3(row):
    if row['ISO_A3_EH'] in CHINA_ISO3:
        return 'CHN'
    if row['ISO_A3_EH'] != '-99':
        return row['ISO_A3_EH']
    return row['ADM0_A3']

gdf_world_ne['ISO3'] = gdf_world_ne.apply(assign_iso3, axis=1)

gdf_world_iso3_geom = (gdf_world_ne[['ISO3', 'geometry']]
    .dissolve(by='ISO3')
    .reset_index()
 )
gdf_world_iso3_attr = gdf_world_ne.drop(columns='geometry').drop_duplicates(subset='ISO3')
gdf_world_iso3 = gdf_world_iso3_geom.merge(gdf_world_iso3_attr, on='ISO3', how='left')

KEEP_COLS = [
    'ISO3',           # join key
    'geometry',       # geometry
    'NAME',           # country name (for display)
    'NAME_ZH',        # Chinese name (optional)
    'CONTINENT',      # continent: Africa / Asia / Europe / ...
    'REGION_UN',      # UN region (finer)
    'SUBREGION',      # subregion: Eastern Asia / Western Europe / ...
    'REGION_WB',      # World Bank region: East Asia & Pacific / ... (optional)
]

gdf_world_clean = gdf_world_iso3[KEEP_COLS].copy().rename(
    columns={
        'NAME': 'NE_name',
        'NAME_ZH': 'NE_name_zh',
        'CONTINENT': 'continent',
        'REGION_UN': 'NE_region',
        'SUBREGION': 'NE_subregion',
        'REGION_WB': 'NE_region_wb',
    }
)


## Clean FAO data: keep necessary columns and rename

In [ ]:
M49_exclude = [
    "'344", # China, Hong Kong SAR
    "'156", # China, mainland
    "'158", # China, Taiwan Province of
    "'446", # China, Macao SAR
    "'638", # Réunion
    "'097", # European Union (27)
    "'265",
    "'159.03", #China (excluding intra-trade),
    "'530",
    "'736",
] 


In [ ]:
pd.read_csv(cfg["paths"]["fao"] / "2024/production/Production_Crops_Livestock_E_All_Data_NOFLAG.csv", encoding_errors="ignore")\
    .query("`Area Code (M49)` in @M49_exclude")\
    [["Area Code", "Area Code (M49)", "Area"]]\
    .drop_duplicates()

In [ ]:
df_production_all = pd.read_csv(cfg["paths"]["fao"] / "2024/production/Production_Crops_Livestock_E_All_Data_NOFLAG.csv", encoding_errors="ignore")\
    .query("Element == 'Production'")\
    .query("`Area Code (M49)` not in @M49_exclude")\
    .query("`Area Code`<5000")\
    .query("Unit != '1000 No'")\
    .rename(columns={"Item Code (CPC)": "code"})\
    .assign(code=lambda x: x["code"].str.replace("'", ""))\

col_sel = ["Area Code", "Area Code (M49)", "Area", "Item Code", "code", "Item", "Element Code", "Element", "Unit"]
col_sel = [*col_sel, *[f"Y{year}" for year in range(2000, 2025)]]
df_FAO1 = (
    df_production_all[["Area Code", "Area Code (M49)", "Area"]]
    .drop_duplicates()
    .reset_index(drop=True)
    .assign(ISO3=lambda x: cc.convert(x['Area Code'].tolist(), src='FAO', to='ISO3'))
)


In [ ]:
df_FAO2 = pd.read_csv(cfg["paths"]["fao"]  / "2024/trade/Trade_CropsLivestock_E_All_Data_NOFLAG.csv", encoding_errors="ignore")\
    .query("`Area Code`<5000")\
    .query("`Area Code (M49)` not in @M49_exclude")\
    .query("~`Area Code (M49)`.str.startswith(\"'097.\")")\
    .rename(columns={"Item Code (CPC)": "code"})\
    .assign(code=lambda x: x["code"].str.replace("'", ""))\
    .query('~code.str.startswith("F")')\
    [["Area Code", "Area Code (M49)", "Area"]]\
    .drop_duplicates()\
    .reset_index(drop=True)\
    .assign(ISO3=lambda x: cc.convert(x['Area Code'].tolist(), src='FAO', to='ISO3'))


In [ ]:
df_FAO3 = pd.read_csv(cfg["paths"]["fao"]  / "2024/Forestry/Forestry_E_All_Data_NOFLAG.csv", encoding_errors="ignore")\
    .query("`Area Code`<5000")\
    .query("`Area Code (M49)` not in @M49_exclude")\
    .query("~`Area Code (M49)`.str.startswith(\"'097.\")")\
    [["Area Code", "Area Code (M49)", "Area"]]\
    .drop_duplicates()\
    .reset_index(drop=True)\
    .assign(ISO3=lambda x: cc.convert(x['Area Code'].tolist(), src='FAO', to='ISO3'))


In [ ]:
df_FAO = pd.concat([df_FAO1, df_FAO2, df_FAO3], ignore_index=True)\
    .drop_duplicates(subset=['Area Code', 'Area Code (M49)', 'Area', 'ISO3'])\
    .reset_index(drop=True)

### French overseas departments

In [ ]:
not_found_codes = {15, 51, 62, 186, 276, 206, 228, 248}
france_codes = {69, 87, 135, 182}
not_found_codes = not_found_codes | france_codes
print(df_FAO[
    df_FAO['Area Code'].isin(not_found_codes)
])
# dissolved after 2010, just drop

In [ ]:
df_FAO_clean = (
    df_FAO
        .query("`Area Code` not in @not_found_codes")
        .reset_index(drop=True)
        .rename(columns={"Area Code": "FAO_code", "Area Code (M49)": "FAO_m49", "Area": "FAO_name"})
)


## Merge FAO data with the world map

In [ ]:
gdf_world_merged = gdf_world_clean.merge(df_FAO_clean, on='ISO3', how='outer')
gdf_world_merged.query("NE_name.isna()")

In [ ]:
gdf_world_merged = gdf_world_merged.dropna(subset=['NE_name']).reset_index(drop=True)

In [ ]:
SUBREGION_TO_PNAS = {
    # Northern Africa and Western Asia
    'Northern Africa':          'N. Africa W. Asia',
    'Western Asia':             'N. Africa W. Asia',

    # Sub-Saharan Africa
    'Middle Africa':            'Sub-Saharan Africa',
    'Eastern Africa':           'Sub-Saharan Africa',
    'Western Africa':           'Sub-Saharan Africa',
    'Southern Africa':          'Sub-Saharan Africa',

    # Central Asia and Russian Federation
    'Central Asia':             'C. Asia',

    # Eastern Asia
    'Eastern Asia':             'E. Asia',

    # Southern Asia
    'Southern Asia':            'S. Asia',

    # Southeastern Asia
    'South-Eastern Asia':       'S. Asia',

    # Northern America
    'Northern America':         'N. America',

    # Latin America and the Caribbean
    'South America':            'Latin America',
    'Central America':          'Latin America',
    'Caribbean':                'Latin America',

    # Western Europe
    'Western Europe':           'W. Europe',
    'Northern Europe':          'W. Europe',
    'Southern Europe':          'W. Europe',

    # Eastern and South-Eastern Europe
    'Eastern Europe':           'E. Europe',

    # Oceania
    'Australia and New Zealand': 'Oceania',
    'Melanesia':                'Oceania',
    'Micronesia':               'Oceania',
    'Polynesia':                'Oceania',

    # no match
    'Seven seas (open ocean)':  None,
}

gdf_world_merged['PNAS_region'] = gdf_world_merged['NE_subregion'].map(SUBREGION_TO_PNAS)


In [ ]:
# handle Russia separately
gdf_world_merged.loc[gdf_world_merged['ISO3'] == 'RUS', 'PNAS_region'] = 'C. Asia'

## Geodesic (spherical) area

In [ ]:
from geocube.api.core import make_geocube
from grid_utils import create_target_grid
da_template = create_target_grid()

In [ ]:
da_countries = make_geocube(gdf_world_merged, measurements=["idx"], like=da_template)["idx"]

In [ ]:
from pyproj import Geod

geod = Geod(ellps='WGS84')

resolution = 0.1
half_res = resolution/2

lat = np.arange(-60 + half_res, 80, resolution)  # avoid polar issues
lon = np.arange(-180 + half_res, 180, resolution)

area_grid = np.zeros((len(lat), len(lon)))

for i, lat_center in enumerate(lat):
    poly_area, poly_perimeter = geod.polygon_area_perimeter(
        [0, resolution, resolution, 0, 0],  # longitude vertices
        [lat_center-half_res, lat_center-half_res, lat_center+half_res, lat_center+half_res, lat_center-0.05]  # latitude vertices
    )
    area_grid[i, :] = abs(poly_area)

da_pixel_area = xr.DataArray(
    area_grid,
    dims=('y', 'x'),
    coords={'y': lat, 'x': lon},
    attrs={'units': 'km²'},
    name='pixel_area'
).rio.write_crs("EPSG:4326")\
    .rio.reproject_match(da_countries)
    
dic_area = pd.DataFrame({
    "idx": da_countries.values.flatten(),
    "area": da_pixel_area.values.flatten()
}).groupby("idx", as_index=False)["area"].sum()\
    .assign(ISO3=lambda _df: _df["idx"].map(gdf_world_merged.set_index("idx")["ISO3"].to_dict()))\
    .groupby("ISO3", as_index=False)["area"].sum()\
    .set_index("ISO3")["area"].to_dict()


In [ ]:
gdf_world_merged = gdf_world_merged.assign(spheric_area=lambda df: df['ISO3'].map(dic_area))


In [ ]:
gdf_world_merged.to_parquet(cfg["paths"]["output"] / "world_clean.parquet")
